In [ ]:
# %% [markdown]
# # Otimização de MFCC para Classificação de Sons Ambientais

# %% [markdown]
# ## Importação das bibliotecas necessárias

# %%
import librosa
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

# %% [markdown]
# ## Configuração inicial e definição dos parâmetros

# %%
# Lista de configurações de MFCC para teste
configs = [
    {'n_mfcc': 12, 'hop_length': 512, 'n_fft': 2048, 'window': 'hann'},   # Configuração baseline
    {'n_mfcc': 13, 'hop_length': 512, 'n_fft': 2048, 'window': 'hann'},   # Mais coeficientes
    {'n_mfcc': 20, 'hop_length': 512, 'n_fft': 2048, 'window': 'hann'},   # Muitos coeficientes
    {'n_mfcc': 13, 'hop_length': 256, 'n_fft': 2048, 'window': 'hann'},   # Alta resolução temporal
    {'n_mfcc': 13, 'hop_length': 1024, 'n_fft': 2048, 'window': 'hann'}   # Baixa resolução temporal
]

# Defina o caminho para suas amostras de áudio
# Estrutura de pastas sugerida: dados_audio/[categoria]/*.wav
caminho_base = "dados_audio"  # Altere para seu caminho

# Categorias de sons ambientais
categorias = ['traffic', 'nature', 'construction', 'human_activity', 'emergency']

# %% [markdown]
# ## Funções auxiliares

# %%
def carregar_amostras(caminho_base, categorias, max_amostras=3):
    """
    Carrega amostras de áudio de cada categoria
    """
    dados_audio = {}
    
    for categoria in categorias:
        caminho_categoria = os.path.join(caminho_base, categoria)
        arquivos = []
        
        # Verifica se a pasta existe
        if os.path.exists(caminho_categoria):
            # Pega até max_amostras arquivos da categoria
            for arquivo in os.listdir(caminho_categoria)[:max_amostras]:
                if arquivo.endswith(('.wav', '.mp3')):
                    caminho_completo = os.path.join(caminho_categoria, arquivo)
                    try:
                        y, sr = librosa.load(caminho_completo, sr=None, duration=5)  # Carrega 5 segundos
                        arquivos.append({
                            'nome': arquivo,
                            'audio': y,
                            'sr': sr
                        })
                        print(f"Carregado: {categoria}/{arquivo}")
                    except Exception as e:
                        print(f"Erro ao carregar {arquivo}: {e}")
        
        dados_audio[categoria] = arquivos
    
    return dados_audio

# %%
def extrair_mfcc_com_estatisticas(audio_data, config):
    """
    Extrai MFCC e calcula estatísticas básicas
    """
    y = audio_data['audio']
    sr = audio_data['sr']
    
    # Extrai MFCC
    mfccs = librosa.feature.mfcc(y=y, sr=sr, **config)
    
    # Calcula estatísticas por coeficiente
    stats = {
        'mean': np.mean(mfccs, axis=1),
        'std': np.std(mfccs, axis=1),
        'min': np.min(mfccs, axis=1),
        'max': np.max(mfccs, axis=1)
    }
    
    return mfccs, stats

# %% [markdown]
# ## Step 1: Extração e análise baseline

# %%
# Carrega as amostras de áudio
print("Carregando amostras de áudio...")
dados_audio = carregar_amostras(caminho_base, categorias)

# %%
# Dicionário para armazenar resultados
resultados_mfcc = {}

# Para cada configuração
for i, config in enumerate(configs):
    config_name = f"Config_{i+1}_n{config['n_mfcc']}_h{config['hop_length']}"
    resultados_mfcc[config_name] = {}
    
    print(f"\n--- {config_name} ---")
    
    # Para cada categoria
    for categoria in categorias:
        if categoria not in dados_audio or len(dados_audio[categoria]) == 0:
            continue
            
        resultados_mfcc[config_name][categoria] = []
        
        # Para cada amostra na categoria
        for amostra in dados_audio[categoria]:
            # Extrai MFCC
            mfccs, stats = extrair_mfcc_com_estatisticas(amostra, config)
            
            # Armazena resultados
            resultados_mfcc[config_name][categoria].append({
                'mfccs': mfccs,
                'stats': stats,
                'shape': mfccs.shape
            })
            
            print(f"{categoria}: {mfccs.shape} - {mfccs.shape[1]} frames temporais")

# %% [markdown]
# ### Visualização comparativa das diferentes configurações

# %%
def plot_mfcc_comparativo(resultados_mfcc, categoria_exemplo='traffic'):
    """
    Cria visualização comparativa dos MFCCs para uma categoria específica
    """
    n_configs = len(resultados_mfcc)
    fig, axes = plt.subplots(n_configs, 1, figsize=(12, 3*n_configs))
    
    for idx, (config_name, dados_categorias) in enumerate(resultados_mfcc.items()):
        if categoria_exemplo in dados_categorias and len(dados_categorias[categoria_exemplo]) > 0:
            # Pega primeira amostra da categoria
            mfccs = dados_categorias[categoria_exemplo][0]['mfccs']
            
            # Plota espectrograma MFCC
            img = librosa.display.specshow(mfccs, x_axis='time', ax=axes[idx])
            axes[idx].set_title(f'{config_name} - {categoria_exemplo}')
            axes[idx].set_ylabel('MFCC Coef')
            plt.colorbar(img, ax=axes[idx], format='%+2.0f')
    
    plt.tight_layout()
    plt.show()

# Plot para cada categoria
for categoria in categorias:
    if categoria in dados_audio and len(dados_audio[categoria]) > 0:
        plot_mfcc_comparativo(resultados_mfcc, categoria)

# %% [markdown]
# ## Step 2: Análise de discriminação entre classes

# %%
def analisar_separacao_classes(resultados_mfcc):
    """
    Calcula distâncias entre as médias dos MFCCs de diferentes categorias
    """
    resultados_distancia = {}
    
    for config_name, dados_categorias in resultados_mfcc.items():
        # Calcula vetores médios para cada categoria
        medias_categorias = {}
        
        for categoria in dados_categorias:
            if len(dados_categorias[categoria]) > 0:
                # Média de todas as amostras da categoria
                mfccs_amostras = []
                for amostra in dados_categorias[categoria]:
                    # Usa a média temporal dos MFCCs
                    mfccs_amostras.append(np.mean(amostra['mfccs'], axis=1))
                
                if mfccs_amostras:
                    medias_categorias[categoria] = np.mean(mfccs_amostras, axis=0)
        
        # Calcula matriz de distâncias entre categorias
        categorias_lista = list(medias_categorias.keys())
        n_categorias = len(categorias_lista)
        
        if n_categorias >= 2:
            matriz_distancias = np.zeros((n_categorias, n_categorias))
            
            for i, cat_i in enumerate(categorias_lista):
                for j, cat_j in enumerate(categorias_lista):
                    if i < j:
                        dist = np.linalg.norm(medias_categorias[cat_i] - medias_categorias[cat_j])
                        matriz_distancias[i, j] = dist
                        matriz_distancias[j, i] = dist
            
            resultados_distancia[config_name] = {
                'categorias': categorias_lista,
                'matriz': matriz_distancias,
                'distancia_media': np.mean(matriz_distancias[matriz_distancias > 0])
            }
            
            print(f"\n{config_name} - Distância média entre classes: {resultados_distancia[config_name]['distancia_media']:.2f}")
    
    return resultados_distancia

# %%
# Análise de separação
resultados_distancia = analisar_separacao_classes(resultados_mfcc)

# %%
# Análise de variância intra-classe
def analise_variancia_intraclasse(resultados_mfcc):
    """
    Calcula variância dentro de cada categoria
    """
    for config_name, dados_categorias in resultados_mfcc.items():
        print(f"\n--- {config_name} ---")
        
        for categoria in dados_categorias:
            if len(dados_categorias[categoria]) > 1:
                # Coleta médias temporais de todas as amostras
                medias_amostras = []
                for amostra in dados_categorias[categoria]:
                    medias_amostras.append(np.mean(amostra['mfccs'], axis=1))
                
                # Calcula variância entre amostras da mesma categoria
                variancia = np.var(medias_amostras, axis=0)
                print(f"{categoria}: variância média = {np.mean(variancia):.2f}")

# Executa análise
analise_variancia_intraclasse(resultados_mfcc)

# %% [markdown]
# ## Step 3: Avaliação de eficiência computacional

# %%
def avaliar_tempo_extracao(dados_audio, configs, n_repeticoes=3):
    """
    Mede tempo de extração para diferentes configurações
    """
    resultados_tempo = []
    
    # Pega uma amostra para teste
    amostra_teste = None
    for categoria in dados_audio:
        if len(dados_audio[categoria]) > 0:
            amostra_teste = dados_audio[categoria][0]
            break
    
    if amostra_teste is None:
        print("Nenhuma amostra encontrada para teste")
        return resultados_tempo
    
    y = amostra_teste['audio']
    sr = amostra_teste['sr']
    
    print(f"\n--- Análise de tempo de extração ---")
    print(f"Arquivo de teste: duração = {len(y)/sr:.2f}s, sr = {sr}Hz")
    
    for i, config in enumerate(configs):
        tempos = []
        
        for _ in range(n_repeticoes):
            start_time = time.time()
            mfccs = librosa.feature.mfcc(y=y, sr=sr, **config)
            end_time = time.time()
            tempos.append(end_time - start_time)
        
        tempo_medio = np.mean(tempos)
        shape = mfccs.shape
        memoria_estimada = shape[0] * shape[1] * 4 / (1024 * 1024)  # MB (assumindo float32)
        
        resultado = {
            'config': f"Config_{i+1}_n{config['n_mfcc']}_h{config['hop_length']}",
            'tempo_medio_ms': tempo_medio * 1000,
            'n_coeficientes': shape[0],
            'n_frames': shape[1],
            'memoria_MB': memoria_estimada
        }
        resultados_tempo.append(resultado)
        
        print(f"{resultado['config']}: {resultado['tempo_medio_ms']:.1f}ms, "
              f"{resultado['n_frames']} frames, {resultado['memoria_MB']:.2f}MB")
    
    return resultados_tempo

# %%
# Executa avaliação de tempo
resultados_tempo = avaliar_tempo_extracao(dados_audio, configs)

# %%
# Estimativa para processamento em larga escala
def estimar_escalabilidade(resultados_tempo, n_arquivos=1000):
    """
    Estima requisitos para processamento de múltiplos arquivos
    """
    print(f"\n--- Estimativa para processamento de {n_arquivos} arquivos ---")
    
    for resultado in resultados_tempo:
        tempo_total = resultado['tempo_medio_ms'] * n_arquivos / 1000  # segundos
        memoria_total = resultado['memoria_MB'] * n_arquivos / 1024  # GB
        
        print(f"\n{resultado['config']}:")
        print(f"  Tempo total: {tempo_total/60:.1f} minutos")
        print(f"  Memória total: {memoria_total:.1f}GB")

estimar_escalabilidade(resultados_tempo)

# %% [markdown]
# ## Resumo e Recomendações

# %%
def gerar_recomendacoes(resultados_distancia, resultados_tempo):
    """
    Gera recomendações baseadas nas análises
    """
    print("\n" + "="*60)
    print("RECOMENDAÇÕES PARA IMPLEMENTAÇÃO EM PRODUÇÃO")
    print("="*60)
    
    # Ordena configurações por distância entre classes (maior é melhor)
    if resultados_distancia:
        configs_ordenadas = sorted(resultados_distancia.items(), 
                                  key=lambda x: x[1]['distancia_media'], 
                                  reverse=True)
        
        print("\n📊 Melhores configurações por separação entre classes:")
        for config_name, dados in configs_ordenadas[:3]:
            print(f"  • {config_name}: distância média = {dados['distancia_media']:.2f}")
    
    # Análise de trade-offs
    print("\n⚖️ Trade-offs identificados:")
    print("  • Mais coeficientes MFCC: melhor separação, porém maior dimensionalidade")
    print("  • Hop length menor: melhor resolução temporal, porém mais dados")
    print("  • Hop length maior: mais eficiente, porém pode perder eventos rápidos")
    
    # Recomendação final
    print("\n✅ RECOMENDAÇÃO FINAL:")
    print("""
    Para classificação de sons ambientais em produção:
    
    - Use n_mfcc=13 (bom equilíbrio entre informação e dimensionalidade)
    - Hop length=512 (resolução temporal adequada para eventos urbanos)
    - Considere hop_length=256 se eventos muito rápidos forem críticos
    
    Esta configuração oferece:
    • Boa separação entre classes de sons ambientais
    • Dimensionalidade gerenciável para processamento em larga escala
    • Resolução temporal adequada para a maioria dos eventos urbanos
    """)

# Gera recomendações
gerar_recomendacoes(resultados_distancia, resultados_tempo)

# %% [markdown]
# ## Conclusão
# 
# O código acima demonstra:
# 1. Extração de MFCC com diferentes parâmetros
# 2. Análise visual comparativa
# 3. Avaliação quantitativa da separação entre classes
# 4. Medição de eficiência computacional
# 5. Recomendações baseadas em evidências para produção